# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [36]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [37]:
spaceship_df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Perform Train Test Split**

In [38]:
X = spaceship_df.drop(columns=["Transported"])
y = spaceship_df["Transported"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.copy()
X_test = X_test.copy()


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [39]:
# Feature selection
X_train = X_train.drop(columns=["PassengerId", "Name"])
X_test = X_test.drop(columns=["PassengerId", "Name"])



# Cabin engineering
X_train["Cabin"] = X_train["Cabin"].str[0]
X_test["Cabin"] = X_test["Cabin"].str[0]



In [40]:
# Fill missing values
for col in X_train.columns:
    if X_train[col].dtype == "object":
        X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
        X_test[col] = X_test[col].fillna(X_test[col].mode()[0])
    else:
        X_train[col] = X_train[col].fillna(X_train[col].median())
        X_test[col] = X_test[col].fillna(X_test[col].median())


/var/folders/xg/zchszbrn7h32p8rl8v_7dg4w0000gn/T/ipykernel_83576/2604094071.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
/var/folders/xg/zchszbrn7h32p8rl8v_7dg4w0000gn/T/ipykernel_83576/2604094071.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test[col] = X_test[col].fillna(X_test[col].mode()[0])


In [41]:
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

scaler = MinMaxScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [42]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("Decision Tree train:", dt.score(X_train, y_train))
print("Decision Tree test:", dt.score(X_test, y_test))



Decision Tree train: 0.9391716997411562
Decision Tree test: 0.7429557216791259


- Bagging and Pasting

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42,
    bootstrap=True)

bagging.fit(X_train, y_train)

print("Bagging train accuracy:", bagging.score(X_train, y_train))
print("Bagging test accuracy:", bagging.score(X_test, y_test))


Bagging train accuracy: 0.9391716997411562
Bagging test accuracy: 0.7745830937320299


- Random Forests

In [44]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Random Forest train accuracy:", rf.score(X_train, y_train))
print("Random Forest test accuracy:", rf.score(X_test, y_test))

Random Forest train accuracy: 0.9391716997411562
Random Forest test accuracy: 0.780333525014376


- Gradient Boosting

In [45]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

print("Gradient Boosting train accuracy:", gb.score(X_train, y_train))
print("Gradient Boosting test accuracy:", gb.score(X_test, y_test))

Gradient Boosting train accuracy: 0.8149266609145815
Gradient Boosting test accuracy: 0.7901092581943646


- Adaptive Boosting

In [48]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
    n_estimators=100,
    random_state=42)

ada.fit(X_train, y_train)

print("AdaBoost train accuracy:", ada.score(X_train, y_train))
print("AdaBoost test accuracy:", ada.score(X_test, y_test))

AdaBoost train accuracy: 0.7966637906241012
AdaBoost test accuracy: 0.78205865439908


Which model is the best and why?


**The best model was Gradient Boosting because it achieved the highest test accuracy after we handled the missing values. It generalized better than the other models, so it performed best on unseen data.**

